# Systems of Equations - Exercises

12 exercises covering [notes.md](notes.md) and [theory.ipynb](theory.ipynb).

**Format**:
- Problem: mathematical statement plus AI context when useful
- Your Solution: scaffold cell for your work
- Solution: one full worked answer with checks

| Level | Meaning | Exercises |
|---|---|---|
| * | Core mechanics | 1-4 |
| ** | Applied linear systems | 5-9 |
| *** | Optimization / AI systems | 10-12 |

| Topic | Exercises |
|---|---|
| Rank and solution types | 1, 2 |
| Elimination and substitution | 3, 4 |
| Least squares and conditioning | 5, 6 |
| Iterative methods | 7, 8 |
| Nonlinear and constrained systems | 9, 10 |
| AI system views | 11, 12 |

In [ ]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)


def header(title):
    print("\n" + "=" * len(title))
    print(title)
    print("=" * len(title))


def check_close(name, got, expected, tol=1e-8):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} - {name}")
    if not ok:
        print('expected:', expected)
        print('got     :', got)
    return ok


def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} - {name}")
    return cond

---

## Exercise 1: Classify the System Type *

For a system $Ax=b$, classify it as `unique`, `infinite`, or `inconsistent` using the rank conditions.

**Task**:
- Implement `classify_system(A, b)` using rank logic
- Test it on the three canonical cases

**What this is training**: reading solution structure from algebra before trying to solve numerically.

In [ ]:
# Exercise 1: Your Solution

def classify_system(A, b):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float).reshape(-1, 1)
    # YOUR CODE HERE
    pass


A_unique = np.array([[1, 2], [3, 4]], dtype=float)
b_unique = np.array([5, 11], dtype=float)

A_inf = np.array([[1, 2], [2, 4]], dtype=float)
b_inf = np.array([3, 6], dtype=float)

A_none = np.array([[1, 1], [1, 1]], dtype=float)
b_none = np.array([2, 3], dtype=float)

print(classify_system(A_unique, b_unique))
print(classify_system(A_inf, b_inf))
print(classify_system(A_none, b_none))

In [ ]:
# Exercise 1: Solution

def classify_system(A, b):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float).reshape(-1, 1)
    aug = np.hstack([A, b])
    rank_A = np.linalg.matrix_rank(A)
    rank_aug = np.linalg.matrix_rank(aug)
    n = A.shape[1]

    if rank_A < rank_aug:
        return 'inconsistent'
    if rank_A == n:
        return 'unique'
    return 'infinite'


header('Exercise 1 checks')
check_true('unique case', classify_system(A_unique, b_unique) == 'unique')
check_true('infinite case', classify_system(A_inf, b_inf) == 'infinite')
check_true('inconsistent case', classify_system(A_none, b_none) == 'inconsistent')
print('\nTakeaway: rank(A) versus rank([A|b]) is the full classification rule.')

---

## Exercise 2: Row Reduction to RREF *

Implement a small RREF routine for augmented matrices.

**Task**:
- write `rref(M)`
- return both the reduced matrix and the pivot columns
- apply it to an underdetermined system and identify free columns

**Why this matters**: RREF turns an equation system into an explicit map from pivots to free variables.

In [ ]:
# Exercise 2: Your Solution

def rref(M, tol=1e-12):
    M = np.array(M, dtype=float).copy()
    # YOUR CODE HERE
    pass


M = np.array([
    [1, 2, 0, 1, 1],
    [2, 4, 1, 3, 3],
    [0, 0, 1, 1, 1],
], dtype=float)

R, pivots = rref(M)
print(R)
print('pivots:', pivots)

In [ ]:
# Exercise 2: Solution

def rref(M, tol=1e-12):
    M = np.array(M, dtype=float).copy()
    rows, cols = M.shape
    pivot_cols = []
    r = 0

    for c in range(cols):
        if r >= rows:
            break
        pivot = r + np.argmax(np.abs(M[r:, c]))
        if abs(M[pivot, c]) < tol:
            continue
        if pivot != r:
            M[[r, pivot]] = M[[pivot, r]]
        M[r] = M[r] / M[r, c]
        for i in range(rows):
            if i != r and abs(M[i, c]) > tol:
                M[i] -= M[i, c] * M[r]
        M[np.abs(M) < tol] = 0.0
        pivot_cols.append(c)
        r += 1

    return M, pivot_cols


R, pivots = rref(M)
free_cols = [j for j in range(M.shape[1] - 1) if j not in pivots]

header('Exercise 2 checks')
print('RREF([A|b]) =\n', R)
print('pivot columns:', pivots)
print('free columns :', free_cols)
check_true('two pivot columns', pivots == [0, 2])
check_true('free columns are 1 and 3', free_cols == [1, 3])
print('\nTakeaway: pivot structure, not raw shape, determines free-variable count.')

---

## Exercise 3: Particular Solution Plus Null Space *

Use the system from Exercise 2 and recover:

- one particular solution $x_p$
- a basis for $\operatorname{null}(A)$
- one other exact solution of the form $x_p + x_h$

**Goal**: make the formula `general solution = particular + homogeneous` concrete.

In [ ]:
# Exercise 3: Your Solution

A = np.array([
    [1, 2, 0, 1],
    [2, 4, 1, 3],
    [0, 0, 1, 1],
], dtype=float)
b = np.array([1, 3, 1], dtype=float)

def nullspace(A, tol=1e-12):
    # YOUR CODE HERE
    pass


x_p = np.linalg.lstsq(A, b, rcond=None)[0]
N = nullspace(A)
print('x_p =', x_p)
print('nullspace basis =\n', N)

In [ ]:
# Exercise 3: Solution

def nullspace(A, tol=1e-12):
    U, s, Vt = np.linalg.svd(A)
    rank = np.sum(s > tol)
    return Vt[rank:].T


x_p = np.linalg.lstsq(A, b, rcond=None)[0]
N = nullspace(A)
coeffs = np.array([0.5, -1.25])[:N.shape[1]]
x_alt = x_p + N @ coeffs

header('Exercise 3 checks')
print('particular solution x_p =', x_p)
print('nullspace basis =\n', N)
print('alternative exact solution =', x_alt)
check_close('A x_p = b', A @ x_p, b)
check_close('A x_alt = b', A @ x_alt, b)
check_true('rank + nullity = 4', np.linalg.matrix_rank(A) + N.shape[1] == 4)
print('\nTakeaway: all exact solutions differ by a null-space direction.')

---

## Exercise 4: Forward and Back Substitution *

Implement the two cheap inner-loop solvers used everywhere in LU and Cholesky workflows.

**Task**:
- `forward_substitution(L, b)` for lower triangular systems
- `back_substitution(U, b)` for upper triangular systems

**Why this matters**: factorization is the expensive part; triangular solves are the reusable part.

In [ ]:
# Exercise 4: Your Solution

def forward_substitution(L, b):
    L = np.array(L, dtype=float)
    b = np.array(b, dtype=float)
    # YOUR CODE HERE
    pass


def back_substitution(U, b):
    U = np.array(U, dtype=float)
    b = np.array(b, dtype=float)
    # YOUR CODE HERE
    pass


L = np.array([[2, 0, 0], [1, 3, 0], [4, -2, 1]], dtype=float)
U = np.array([[2, 1, -1], [0, 3, 2], [0, 0, 4]], dtype=float)
b1 = np.array([2, 7, 5], dtype=float)
b2 = np.array([1, 8, 12], dtype=float)

print(forward_substitution(L, b1))
print(back_substitution(U, b2))

In [ ]:
# Exercise 4: Solution

def forward_substitution(L, b):
    L = np.array(L, dtype=float)
    b = np.array(b, dtype=float)
    x = np.zeros_like(b)
    for i in range(len(b)):
        x[i] = (b[i] - L[i, :i] @ x[:i]) / L[i, i]
    return x


def back_substitution(U, b):
    U = np.array(U, dtype=float)
    b = np.array(b, dtype=float)
    x = np.zeros_like(b)
    for i in range(len(b) - 1, -1, -1):
        x[i] = (b[i] - U[i, i + 1:] @ x[i + 1:]) / U[i, i]
    return x


x_forward = forward_substitution(L, b1)
x_back = back_substitution(U, b2)

header('Exercise 4 checks')
check_close('forward substitution', L @ x_forward, b1)
check_close('back substitution', U @ x_back, b2)
print('x_forward =', x_forward)
print('x_back    =', x_back)
print('\nTakeaway: direct solvers are built from these small triangular kernels.')

---

## Exercise 5: Gaussian Elimination with Partial Pivoting **

Write a solver for a square system using elimination plus back substitution.

**Task**:
- perform partial pivoting each column
- eliminate below the pivot
- recover the solution by back substitution

**AI connection**: this is the direct-solve primitive behind many exact medium-size linear algebra workflows.

In [ ]:
# Exercise 5: Your Solution

def gaussian_elimination(A, b, tol=1e-12):
    A = np.array(A, dtype=float).copy()
    b = np.array(b, dtype=float).copy()
    # YOUR CODE HERE
    pass


A5 = np.array([[1., 2., 1.], [2., -1., 3.], [3., 1., -1.]])
b5 = np.array([9., 8., 3.])
print(gaussian_elimination(A5, b5))

In [ ]:
# Exercise 5: Solution

def gaussian_elimination(A, b, tol=1e-12):
    A = np.array(A, dtype=float).copy()
    b = np.array(b, dtype=float).copy()
    n = len(b)

    for k in range(n - 1):
        pivot = k + np.argmax(np.abs(A[k:, k]))
        if abs(A[pivot, k]) < tol:
            raise ValueError('matrix is singular to working precision')
        if pivot != k:
            A[[k, pivot]] = A[[pivot, k]]
            b[[k, pivot]] = b[[pivot, k]]
        for i in range(k + 1, n):
            m = A[i, k] / A[k, k]
            A[i, k:] -= m * A[k, k:]
            b[i] -= m * b[k]

    return back_substitution(A, b)


x5 = gaussian_elimination(A5, b5)
header('Exercise 5 checks')
print('solution =', x5)
check_close('A x = b', A5 @ x5, b5)
check_close('matches np.linalg.solve', x5, np.linalg.solve(A5, b5))
print('\nTakeaway: elimination becomes a practical solver only once pivoting is handled carefully.')

---

## Exercise 6: Least Squares and Ridge **

Fit a line to noisy data with both ordinary least squares and ridge regression.

**Task**:
- solve the normal equations
- compute the residual
- verify orthogonality of the residual to the column space
- compare with ridge regularization

**What to notice**: least squares is projection; ridge changes both geometry and conditioning.

In [ ]:
# Exercise 6: Your Solution

X = np.array([[1, 1], [1, 2], [1, 3], [1, 4]], dtype=float)
y = np.array([2, 3, 3.5, 5], dtype=float)
lam = 0.5

# YOUR CODE HERE
# Solve for w_ols, residual, and w_ridge
pass

In [ ]:
# Exercise 6: Solution

XtX = X.T @ X
Xty = X.T @ y
w_ols = np.linalg.solve(XtX, Xty)
residual = y - X @ w_ols
w_ridge = np.linalg.solve(XtX + lam * np.eye(X.shape[1]), Xty)

header('Exercise 6 checks')
print('OLS solution   =', w_ols)
print('Ridge solution =', w_ridge)
print('Residual       =', residual)
check_close('X^T residual = 0', X.T @ residual, np.zeros(X.shape[1]))
check_true('ridge shrinks parameter norm', np.linalg.norm(w_ridge) < np.linalg.norm(w_ols))
print('\nTakeaway: OLS finds the projection; ridge trades exact projection for better-controlled parameters.')

---

## Exercise 7: Conditioning and Sensitivity **

A system can be exactly solvable and still be numerically fragile.

**Task**:
- compute the condition number of an ill-conditioned matrix
- solve with one right-hand side
- perturb the right-hand side slightly
- measure the amplification in the solution

**AI connection**: this is the same phenomenon behind unstable Hessian solves and fragile kernel systems.

In [ ]:
# Exercise 7: Your Solution

A = np.array([[1000., 999.], [999., 998.]], dtype=float)
b = np.array([1999., 1997.], dtype=float)
b2 = np.array([1999.001, 1997.], dtype=float)

# YOUR CODE HERE
pass

In [ ]:
# Exercise 7: Solution

condA = np.linalg.cond(A)
x = np.linalg.solve(A, b)
x2 = np.linalg.solve(A, b2)
rhs_rel = np.linalg.norm(b2 - b) / np.linalg.norm(b)
x_rel = np.linalg.norm(x2 - x) / np.linalg.norm(x)

header('Exercise 7 checks')
print('det(A)     =', np.linalg.det(A))
print('cond_2(A)  =', condA)
print('x          =', x)
print('x perturbed=', x2)
print('relative rhs change =', rhs_rel)
print('relative x change   =', x_rel)
print('amplification       =', x_rel / rhs_rel)
check_true('system is ill-conditioned', condA > 1e5)
print('\nTakeaway: exact solvability says nothing about numerical safety.')

---

## Exercise 8: Jacobi and Gauss-Seidel **

Compare two stationary iterative methods on a small SPD system.

**Task**:
- implement `jacobi_steps` and `gauss_seidel_steps`
- run 3 iterations from the zero vector
- record the residual after each step

**What to notice**: using fresh values usually improves convergence.

In [ ]:
# Exercise 8: Your Solution

A = np.array([[4., 1.], [1., 3.]], dtype=float)
b = np.array([1., 2.], dtype=float)
x0 = np.zeros(2)

def jacobi_steps(A, b, x0, steps):
    # YOUR CODE HERE
    pass


def gauss_seidel_steps(A, b, x0, steps):
    # YOUR CODE HERE
    pass


print(jacobi_steps(A, b, x0, 3))
print(gauss_seidel_steps(A, b, x0, 3))

In [ ]:
# Exercise 8: Solution

def jacobi_steps(A, b, x0, steps):
    D = np.diag(np.diag(A))
    R = A - D
    x = x0.copy()
    hist = []
    for _ in range(steps):
        x = np.linalg.solve(D, b - R @ x)
        hist.append((x.copy(), np.linalg.norm(b - A @ x)))
    return hist


def gauss_seidel_steps(A, b, x0, steps):
    x = x0.copy()
    hist = []
    for _ in range(steps):
        for i in range(len(b)):
            x[i] = (b[i] - A[i, :i] @ x[:i] - A[i, i+1:] @ x[i+1:]) / A[i, i]
        hist.append((x.copy(), np.linalg.norm(b - A @ x)))
    return hist


jac_hist = jacobi_steps(A, b, x0, 3)
gs_hist = gauss_seidel_steps(A, b, x0, 3)

header('Exercise 8 checks')
print('Jacobi history:')
for k, (xk, rk) in enumerate(jac_hist, start=1):
    print(f'  iter {k}: x = {xk}, residual = {rk:.6e}')
print('\nGauss-Seidel history:')
for k, (xk, rk) in enumerate(gs_hist, start=1):
    print(f'  iter {k}: x = {xk}, residual = {rk:.6e}')
check_true('Gauss-Seidel residual is smaller by iter 3', gs_hist[-1][1] < jac_hist[-1][1])
print('\nTakeaway: structure-aware reuse of fresh information speeds convergence.')

---

## Exercise 9: Conjugate Gradient **

Run two steps of conjugate gradient on the same SPD system.

**Task**:
- implement a small CG loop
- report iterates and residual norms
- compare with the exact solution

**What this trains**: understanding Krylov methods as optimization over better search directions, not just repeated substitution.

In [ ]:
# Exercise 9: Your Solution

A = np.array([[4., 1.], [1., 3.]], dtype=float)
b = np.array([1., 2.], dtype=float)
x0 = np.zeros(2)

def cg(A, b, x0, steps):
    # YOUR CODE HERE
    pass


print(cg(A, b, x0, 2))

In [ ]:
# Exercise 9: Solution

def cg(A, b, x0, steps):
    x = x0.copy()
    r = b - A @ x
    p = r.copy()
    hist = [(x.copy(), np.linalg.norm(r))]
    for _ in range(steps):
        Ap = A @ p
        alpha = (r @ r) / (p @ Ap)
        x = x + alpha * p
        r_new = r - alpha * Ap
        beta = (r_new @ r_new) / (r @ r)
        p = r_new + beta * p
        r = r_new
        hist.append((x.copy(), np.linalg.norm(r)))
    return hist


hist = cg(A, b, x0, 2)
x_star = np.linalg.solve(A, b)

header('Exercise 9 checks')
for k, (xk, rk) in enumerate(hist):
    print(f'iter {k}: x = {xk}, residual = {rk:.6e}')
check_close('final iterate equals exact solution', hist[-1][0], x_star)
print('\nTakeaway: for small SPD systems, CG can reach the exact answer in very few steps.')

---

## Exercise 10: Newton's Method for a Nonlinear System ***

Consider

$$
F(x,y) = (x^2 + y^2 - 1,\ x - y^2).
$$

**Task**:
- implement `F`, its Jacobian `J`, and two Newton steps from `(1, 1)`
- report each linear solve and the resulting residual norm

**What this is training**: nonlinear solving as repeated local linear solving.

In [ ]:
# Exercise 10: Your Solution

def F(z):
    # YOUR CODE HERE
    pass


def J(z):
    # YOUR CODE HERE
    pass


def newton(z0, steps):
    # YOUR CODE HERE
    pass


print(newton((1.0, 1.0), 2))

In [ ]:
# Exercise 10: Solution

def F(z):
    x, y = z
    return np.array([x**2 + y**2 - 1, x - y**2], dtype=float)


def J(z):
    x, y = z
    return np.array([[2*x, 2*y], [1.0, -2*y]], dtype=float)


def newton(z0, steps):
    z = np.array(z0, dtype=float)
    hist = []
    for _ in range(steps):
        delta = np.linalg.solve(J(z), -F(z))
        z = z + delta
        hist.append((z.copy(), np.linalg.norm(F(z))))
    return hist


hist = newton((1.0, 1.0), 2)
header('Exercise 10 checks')
for k, (zk, fk) in enumerate(hist, start=1):
    print(f'iter {k}: z = {zk}, ||F(z)|| = {fk:.6e}')
check_true('Newton reduced residual', hist[-1][1] < hist[0][1])
print('\nTakeaway: every Newton step is a linear system solve inside a nonlinear problem.')

---

## Exercise 11: KKT Saddle-Point System ***

Minimize $f(x, y) = x^2 + 2y^2$ subject to $x + y = 3$.

**Task**:
- build the KKT matrix explicitly
- solve for $(x, y, \lambda)$
- verify the constraint and objective value

**AI connection**: constrained optimization and KL-regularized objectives lead to the same kind of block system.

In [ ]:
# Exercise 11: Your Solution

# YOUR CODE HERE
pass

In [ ]:
# Exercise 11: Solution

KKT = np.array([
    [2.0, 0.0, 1.0],
    [0.0, 4.0, 1.0],
    [1.0, 1.0, 0.0],
])
rhs = np.array([0.0, 0.0, 3.0])
sol = np.linalg.solve(KKT, rhs)
x, y, lam = sol

header('Exercise 11 checks')
print('solution [x, y, lambda] =', sol)
check_close('constraint x + y = 3', np.array([x + y]), np.array([3.0]))
print('objective value =', x**2 + 2 * y**2)
print('\nTakeaway: constraints enlarge the variable set and create a block system, not a side condition.')

---

## Exercise 12: Attention and GP Systems in AI ***

This final exercise combines two AI views of system solving.

**Part A**: compute a tiny attention score matrix, row-wise softmax weights, and output matrix.

**Part B**: solve a tiny Gaussian-process style SPD system.

**Goal**: see that modern AI often hides classical linear-system structure under different language.

In [ ]:
# Exercise 12: Your Solution

Q = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)
K = np.array([[1, 1], [1, 0], [0, 1]], dtype=float)
V = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)

K_gp = np.array([[1.0, 0.7, 0.4], [0.7, 1.0, 0.6], [0.4, 0.6, 1.0]], dtype=float)
sigma2 = 0.1
y_gp = np.array([1.0, -0.5, 0.75], dtype=float)

# YOUR CODE HERE
pass

In [ ]:
# Exercise 12: Solution

scores = Q @ K.T / np.sqrt(2.0)
weights = np.exp(scores - scores.max(axis=1, keepdims=True))
weights = weights / weights.sum(axis=1, keepdims=True)
O = weights @ V

alpha = np.linalg.solve(K_gp + sigma2 * np.eye(3), y_gp)

header('Exercise 12 checks')
print('attention scores =\n', scores)
print('\nattention weights =\n', weights)
print('row sums =', weights.sum(axis=1))
print('\nattention output =\n', O)
check_close('weights lie on simplex', weights.sum(axis=1), np.ones(weights.shape[0]))
print('\nGP-style alpha =', alpha)
check_close('SPD solve residual', (K_gp + sigma2 * np.eye(3)) @ alpha, y_gp)
print('\nTakeaway: attention solves constrained weighting problems; GP inference solves SPD linear systems.')

---

## Final Solution Checks

This final cell is not a substitute for doing the exercises. It is only a compact regression check for the reference solutions.

In [ ]:
header('Final solution checks')

checks = []
checks.append(classify_system(A_unique, b_unique) == 'unique')
checks.append(classify_system(A_inf, b_inf) == 'infinite')
checks.append(classify_system(A_none, b_none) == 'inconsistent')
checks.append(np.allclose(L @ x_forward, b1))
checks.append(np.allclose(U @ x_back, b2))
checks.append(np.allclose(A5 @ x5, b5))
checks.append(np.allclose(X.T @ residual, np.zeros(X.shape[1])))
checks.append(gs_hist[-1][1] < jac_hist[-1][1])
checks.append(np.allclose(hist[-1][0], np.linalg.solve(np.array([[4., 1.], [1., 3.]]), np.array([1., 2.]))))
checks.append(np.allclose(weights.sum(axis=1), np.ones(weights.shape[0])))
checks.append(np.allclose((K_gp + sigma2 * np.eye(3)) @ alpha, y_gp))

print(f'Passed {sum(checks)} / {len(checks)} checks')
if all(checks):
    print('All reference solutions are internally consistent.')
else:
    print('At least one reference solution needs review.')